# 🤝 Notebook 2: Hinted Handoff

When the coordinator notices a replica is down, instead of dropping the write it **stores a hint** locally: *'when r3 comes back, give it this'*. As soon as r3 reappears, the coordinator drains the hint queue.

Used by Cassandra and DynamoDB-style systems to maintain availability without giving up convergence.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/hinted-handoff
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 Implementation

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Dict

@dataclass
class Replica:
    name: str
    up: bool = True
    data: Dict[str, str] = field(default_factory=dict)
    def write(self, key, value):
        if not self.up: return False
        self.data[key] = value
        return True

class Coordinator:
    def __init__(self, replicas):
        self.replicas = replicas
        self.hints = defaultdict(deque)  # replica_name -> queue of (key, value)

    def write(self, key, value):
        for r in self.replicas:
            if not r.write(key, value):
                self.hints[r.name].append((key, value))
                print(f'  📝 stored hint for {r.name}: ({key}, {value})')

    def deliver_hints(self):
        for r in self.replicas:
            if r.up and self.hints[r.name]:
                count = 0
                while self.hints[r.name]:
                    k,v = self.hints[r.name].popleft()
                    r.write(k,v); count += 1
                print(f'  ✉ delivered {count} hint(s) to {r.name}')

replicas = [Replica('r1'), Replica('r2'), Replica('r3')]
coord = Coordinator(replicas)

replicas[2].up = False
for i in range(5):
    coord.write(f'k{i}', f'v{i}')

print('--- r3 recovers ---')
replicas[2].up = True
coord.deliver_hints()

for r in replicas:
    print(f'{r.name}: {r.data}')


All three replicas converge once `r3` is back, **without** requiring a full anti-entropy scan.

## ⚠️ Caveats

- Hints take **memory and disk** on the coordinator — Cassandra caps how long they're kept (default 3h). After that you must run a full repair.
- A hint is only useful if **the coordinator** itself stays alive. Pair with anti-entropy / Merkle-tree repair for the long tail.
- The order in which hints are replayed matters if writes target the same key — store a per-write timestamp.